In [2]:
# Colab mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
BASE_DIR = Path("/content/drive/MyDrive/ai4trade")
FINAL_DIR = BASE_DIR / "predictions" / "final"
HARMONIZED = BASE_DIR / "data" / "interim" / "final_clean_data.parquet"

# Load harmonized actuals (has Aug for CHN, Jul for USA)
act = pd.read_parquet(HARMONIZED)
act["month"] = pd.to_datetime(act["month"])

# Load your ensemble HS4
ens = pd.read_parquet(FINAL_DIR / "final_forecast_hs4_final.parquet")
ens["month"] = pd.to_datetime("2025-10-01")   # unified forecast month

# Function: sMAPE
def smape(y_true, y_pred, eps=1.0):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    num = np.abs(y_pred - y_true)
    den = (np.abs(y_true) + np.abs(y_pred) + eps) / 2
    return np.mean(num / den)

# --- China check ---
chn_true = act[(act["origin"]=="CHN") & (act["month"]=="2025-08-01")]  # latest available
chn_pred = ens[ens["origin"]=="CHN"]
# merge on destination+hs4+trade_flow
cols = ["destination","trade_flow","hs4"]
df_chn = chn_pred.merge(chn_true, on=cols, suffixes=("_pred","_naive"))
smape_naive_chn = smape(df_chn["value_naive"], df_chn["value_pred"])
print(f"CHN ensemble vs naive carry sMAPE: {smape_naive_chn:.3f}")

# --- USA check ---
usa_true = act[(act["origin"]=="USA") & (act["month"]=="2025-07-01")]
usa_pred = ens[ens["origin"]=="USA"]
df_usa = usa_pred.merge(usa_true, on=cols, suffixes=("_pred","_naive"))
smape_naive_usa = smape(df_usa["value_naive"], df_usa["value_pred"])
print(f"USA ensemble vs naive carry sMAPE: {smape_naive_usa:.3f}")

KeyError: 'value_naive'